# GrandNexus v7.4.0-rc — check on Colab (logbook 68-71)
Two things in this version: the **batch mode** of the benches (`--batch`: same model, same requests, half the price,
asynchronous) and the **first module audition** (ConfidenceEvaluator, offline: it did not pass and stays experimental).
Measured offline before delivery: batch rows identical to the synchronous rows with a scripted API; resume after interruption.
This notebook confirms the batch mode on the real API: **the 5 seen validation items through one batch**, a few cents.

Expected: cell 4 reprints the audition table of logbook 70; cell 5 prints `batch 1 submitted`, waits (usually a few minutes,
up to an hour when the API is busy), then a cost of about $0.03 instead of $0.049; cell 6 prints `tagged v7.4.0`.
Run the cells in order. **If a cell prints STOP, do not run the next ones: send the output.** If cell 5 is interrupted, run it
again: it resumes without submitting twice. If the Colab assistant offers to rewrite a cell, decline. No item text is printed.
Send back the output of cells 5 and 6, and the zip named in cell 6.

## 1. Get exactly v7.4.0-rc

In [ ]:
# 1. Get EXACTLY GrandNexus v7.4.0-rc (never another zip; stop if it is not found)
import os, glob, fnmatch, subprocess, zipfile, shutil, time
VERSION, ZIP_PATTERN = 'v7.4.0-rc', 'GrandNexus_v7.4.0-rc*.zip'
CONTENT, WORK = '/content', '/content/GrandNexus'
MYDRIVE = '/content/drive/MyDrive'

def gn_version(path):
    """(git describe, has the story-aware bench) of a GrandNexus folder, or (None, False)."""
    if not os.path.isdir(os.path.join(path, '.git')):
        return None, False
    d = subprocess.run('git describe --tags', shell=True, cwd=path, capture_output=True, text=True).stdout.strip()
    bench = os.path.join(path, 'tools', 'bench_learning.py')
    ok = all(os.path.exists(os.path.join(path, 'tools', f)) for f in ('colab_save.py', 'cost.py', 'batch_runner.py')) and \
         os.path.exists(bench) and '--story-mode' in open(bench, encoding='utf-8').read()
    return d, ok

try:
    from google.colab import drive; drive.mount(os.path.join(CONTENT, 'drive'))
except Exception as e:
    print('Drive not mounted:', e)

# the release zip, newest first: My Drive/GrandNexus and My Drive first, then anywhere in My Drive (3 levels deep);
# any other GrandNexus zip (older versions, session saves) is ignored
found = glob.glob(os.path.join(MYDRIVE, 'GrandNexus', ZIP_PATTERN)) + glob.glob(os.path.join(MYDRIVE, ZIP_PATTERN))
if not found:
    for root, dirs, names in os.walk(MYDRIVE):
        if root[len(MYDRIVE):].count(os.sep) >= 3:
            dirs[:] = []
        found += [os.path.join(root, n) for n in names if fnmatch.fnmatch(n, ZIP_PATTERN)]
found.sort(key=os.path.getmtime)
if found:
    source = found[-1]
else:
    print(f'{ZIP_PATTERN} not found in My Drive: upload it now (any other zip is refused).')
    try:
        from google.colab import files
        up = next(iter(files.upload()))
    except Exception as e:
        raise SystemExit(f'STOP: no {ZIP_PATTERN} on the Drive and no upload possible ({e}).')
    if not fnmatch.fnmatch(os.path.basename(up), ZIP_PATTERN):
        raise SystemExit(f'STOP: {up} is not {ZIP_PATTERN}.')
    source = os.path.join(CONTENT, up)
print('release zip:', source)

# an existing /content/GrandNexus of another version is moved aside, never reused, never deleted
d, ok = gn_version(WORK)
if os.path.exists(WORK) and not (d or '').startswith(VERSION):
    aside = WORK + '_aside_' + time.strftime('%Y%m%d_%H%M%S')
    shutil.move(WORK, aside); print(f'existing {WORK} ({d}) moved aside to {aside}')
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist() if n.strip('/')}
    z.extractall(CONTENT if top == {'GrandNexus'} else WORK)
    print('extracted')
subprocess.run('git checkout -q main', shell=True, cwd=WORK)
d, ok = gn_version(WORK)
print('git describe:', d, '| v7.4.0-rc tools present:', ok)
print(subprocess.run('git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)
if not ((d or '').startswith(VERSION) and ok):
    raise SystemExit(f'STOP: {WORK} is {d}, not {VERSION}. Do not run the next cells.')
print(f'OK: {VERSION}')


## 2. Packages, key, pre-flight (1 call)

In [ ]:
# 2. Packages, key, pre-flight check (1 call, through the translator itself: the cached system prompt must be accepted)
import subprocess, sys, os
WORK = '/content/GrandNexus'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic', 'clingo', 'sympy', 'networkx', 'psutil'])
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
sys.path.insert(0, WORK)
from grandnexus.llm_interface.translator_v5 import Translator, SYSTEM_TRANSLATE
tr = Translator(model='claude-sonnet-5-5')
reply = tr._call(SYSTEM_TRANSLATE, 'TEXT:\nAll dogs bark. Rex is a dog.\n\nQUESTION (or statement to verify):\nDoes Rex bark?', 'preflight')
r = tr.receipts[-1]
print('reply received:', bool(reply.strip()), '| id:', r.id, '| model:', r.model)
print(f'usage: input {r.tokens_in} | output {r.tokens_out} | cache write {r.cache_write} | cache read {r.cache_read}')
print('the system prompt went into the cache' if (r.cache_write or r.cache_read) else 'WARNING: no cache write or read on this call (send this output)')


## 3. Offline tests (0 calls)

In [ ]:
# 3. Offline tests (0 calls)
import subprocess, sys, glob, os
WORK = '/content/GrandNexus'
bad = []
for t in sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'test_*.py'))):
    r = subprocess.run([sys.executable, '-W', 'ignore', t], cwd=WORK, capture_output=True, text=True)
    last = (r.stdout.strip().splitlines() or ['(no output)'])[-1]
    ok = r.returncode == 0 and not last.startswith('FAIL') and 'Traceback' not in r.stderr
    print(f"{'ok ' if ok else 'BAD'} {os.path.basename(t):40} {last[:70]}")
    if not ok:
        bad.append(t)
if bad:
    raise SystemExit(f'STOP: {len(bad)} offline test(s) failed; send this output.')


## 4. Module audition n°1, replayed (0 calls)

In [ ]:
# 4. Module audition n°1, replayed (0 calls): the table must match logbook 70 (F1 all 0.506, short program decided 0.746)
import subprocess, sys
WORK = '/content/GrandNexus'
r = subprocess.run([sys.executable, '-W', 'ignore', 'tools/audition_confidence.py'], cwd=WORK, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-300:])


## 5. Batch mode on the real API

In [ ]:
# 5. Batch mode on the real API (5 seen validation items = about 6 requests at half price; it waits for the batch: minutes)
import subprocess, sys, glob, json, os
WORK = '/content/GrandNexus'
CMD = 'python tools/bench_folio.py --split validation --n 5 --seed 0 --modes translator --strict --batch'
p = subprocess.Popen(CMD, shell=True, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    sys.stdout.write(line)
p.wait()
res = sorted(glob.glob(os.path.join(WORK, 'tests_lab', 'resultats', 'folio_validation_5_*.json')), key=os.path.getmtime)
if p.returncode != 0 or not res:
    raise SystemExit('STOP: the batch run did not finish. Re-run this cell: it resumes from the saved state. If it fails again, send this output.')
d = json.load(open(res[-1])); rows = d['rows']['translator']; tot = d['summary']['cost']['total']
print(f"\nbatches: {d['summary'].get('batches')} | verdicts: {[r['pred'] for r in rows]} | cost ${tot['usd']:.3f} "
      f"(synchronous with cache in logbook 66: $0.049) | cached share of the input {100 * tot['cache_hit_share']:.0f}%")


## 6. Release decision and save

In [ ]:
# 6. Release decision and save: tag v7.4.0 only if the batch run completed, without error rows, at about half price
import subprocess, sys, glob, json, os
WORK = '/content/GrandNexus'
res = sorted(glob.glob(WORK + '/tests_lab/resultats/folio_validation_5_*.json'), key=os.path.getmtime)
ok = False
if res:
    d = json.load(open(res[-1])); rows = d['rows']['translator']; tot = d['summary'].get('cost', {}).get('total') or {}
    ok = (d['summary'].get('batches', 0) >= 1 and len(rows) == 5 and all(r.get('batch') and r['pred'] != 'error' for r in rows)
          and tot.get('batch') is True and tot.get('usd', 9) <= 0.035)
    print('batch run:', 'PASS' if ok else 'NOT PASS', '| cost', tot.get('usd'), '| limit 0.035 (about half of 0.049, cache hits being best effort)')
git = lambda *a: subprocess.run(['git', *a], cwd=WORK, capture_output=True, text=True).stdout.strip()
git('add', '-A')
git('-c', 'user.name=Khaled', '-c', 'user.email=khaled@grandnexus.local', 'commit', '-qm',
    'v7.4.0 check on Colab: ' + ('batch mode passes on the real API' if ok else 'batch criterion NOT passed'))
if ok:
    git('tag', 'v7.4.0')
    print('tagged v7.4.0')
else:
    print('no release tag: send this output')
r = subprocess.run([sys.executable, 'tools/colab_save.py', '--tag', 'check', '--expect',
                    'tests_lab/resultats/folio_validation_5_*.json', 'tests_lab/resultats/auditions/audition_confidence_*.json'],
                   cwd=WORK, capture_output=True, text=True)
print(r.stdout, r.stderr[-500:])
